In [2]:
import json, hashlib, shutil, random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split

In [18]:
SEED = 42
SRC_ROOT = Path.home() / "datasets" / "IQ-OTHNCCD"
DST = Path("../data")
EXTS = {".jpg", ".jpeg", ".png"}
random.seed(SEED)
np.random.seed(SEED)

In [6]:
def class_of(folder_name: str):
    n = folder_name.lower()
    if "ben" in n:   return "benign"     
    if "malig" in n: return "malignant"
    if "normal" in n: return "normal"
    return None
    
print(class_of("bengin_cases"))
print(class_of("Malignant cases"))
print(class_of("Normal cases"))    
print(class_of("Test cases"))               
    


records = []
for d in SRC_ROOT.rglob("*"):
    if d.is_dir():
        cls = class_of(d.name)
        if cls is None:
            continue
        for f in d.iterdir():
            if f.is_file() and f.suffix.lower() in EXTS:
                records.append({"path": str(f), "class": cls})

df = pd.DataFrame(records)
assert len(df) > 0, "no pic found"
print( ":before removing duplicates\n", df["class"].value_counts(), "\n")

benign
malignant
normal
None
:before removing duplicates
 class
malignant    561
normal       416
benign       120
Name: count, dtype: int64 



In [13]:
def md5(p):
    return hashlib.md5(Path(p).read_bytes()).hexdigest()

df["md5"] = df["path"].apply(md5)
dups = df[df.duplicated("md5", keep="first")]
print(f"removed dups numbers: {len(dups)}")
df = df.drop_duplicates("md5", keep="first").reset_index(drop=True)


sizes, modes = Counter(), Counter()
for p in df["path"]:
    with Image.open(p) as im:
        sizes[im.size] += 1
        modes[im.mode] += 1
print("\n scales:", sizes.most_common(5))
print("colors :", dict(modes))


removed dups numbers: 0

 scales: [((512, 512), 993), ((623, 512), 31), ((801, 512), 28), ((511, 404), 1), ((506, 331), 1)]
colors : {'RGB': 1054}


In [15]:
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["class"], random_state=SEED)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["class"], random_state=SEED)

train_df = train_df.assign(split="train")
val_df = val_df.assign(split="val")
test_df = test_df.assign(split="test")
all_df = pd.concat([train_df, val_df, test_df]).reset_index(drop=True)
all_df.sample(10, random_state=42)

,path,class,md5,split
260,../data/raw/Bengin cases/Bengin case (38).jpg,benign,8d0e3e5e7645465ec0e805cd99f36ba5,train
1025,../data/raw/Malignant cases/Malignant case (39...,malignant,ca9551a8975bbd84b05daf4d6f52c893,test
845,../data/raw/Normal cases/Normal case (149).jpg,normal,9fe280651d19662b526a68a7a9cdcf74,val
1005,../data/raw/Normal cases/Normal case (32).jpg,normal,b4d9ba835ae0cf910be46cce875396e4,test
88,../data/raw/Normal cases/Normal case (135).jpg,normal,751f223793f27ad6752e79f82a384ac2,train
457,../data/raw/Normal cases/Normal case (148).jpg,normal,332744236e00fd1f62a65d04b78820de,train
184,../data/raw/Malignant cases/Malignant case (33...,malignant,b529656f1f341e993f2600c8937fdb79,train
986,../data/raw/Malignant cases/Malignant case (11...,malignant,bdf83d479a5da9672d03bd2025a63c26,test
367,../data/raw/Malignant cases/Malignant case (33...,malignant,2484aae85bd32df8ef69061f633328fb,train
558,../data/raw/Normal cases/Normal case (128).jpg,normal,006e50add470316a5d1a7fff11de337b,train


In [17]:
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    A = set(all_df[all_df.split == a]["md5"]); B = set(all_df[all_df.split == b]["md5"])
    assert not (A & B), f"OVERLAP NETWEEN{a} و {b}!"
print("\n NO DATA LEAKAGE FOUND")



 NO DATA LEAKAGE FOUND
